# NLP Exercises

We have five exercises in this section. The exercises are:
1. Build your own tokenizer, where you need to implement two functions to implement a tokenizer based on regular expression.
2. Get tags from Trump speech.
3. Get the nouns in the last 10 sentences from Trump's speech and find the nouns divided by sentencens. Use SpaCy.
4. Build your own Bag Of Words implementation using tokenizer created before.
5. Build a 5-gram model and clean up the results.

## Exercise 1. Build your own tokenizer

Build two different tokenizers:
- ``tokenize_sentence``: function tokenizing text into sentences,
- ``tokenize_word``: function tokenizing text into words.

In [6]:
from typing import List
import re

def tokenize_words(text: str) -> List[str]:
    """Tokenize text into words using regex.

    Parameters
    ----------
    text: str
            Text to be tokenized

    Returns
    -------
    List[str]
            List containing words tokenized from text

    """
    return re.findall(r"[\w']+", text)

def tokenize_sentence(text: str) -> List[str]:
    """Tokenize text into sentences using regex.

    Parameters
    ----------
    text: str
            Text to be tokenized

    Returns
    -------
    List[str]
            List containing words tokenized from text

    """
    sentences = re.split(r'(?<=[.!?])\s*(?=[A-Z])', text)
    result = []
    for sentence in sentences:
        sentence = sentence.strip()
        if sentence:
            result.append(sentence)
    return result

text = "Here we go again. I was supposed to add this text later.\
Well, it's 10.p.m. here, and I'm actually having fun making this course. :o\
I hope you are getting along fine with this presentation, I really did try.\
And one last sentence, just so you can test you tokenizers better."

print("Tokenized sentences:")
print(tokenize_sentence(text))

print("Tokenized words:")
print(tokenize_words(text))

Tokenized sentences:
['Here we go again.', 'I was supposed to add this text later.', "Well, it's 10.p.m. here, and I'm actually having fun making this course. :oI hope you are getting along fine with this presentation, I really did try.", 'And one last sentence, just so you can test you tokenizers better.']
Tokenized words:
['Here', 'we', 'go', 'again', 'I', 'was', 'supposed', 'to', 'add', 'this', 'text', 'later', 'Well', "it's", '10', 'p', 'm', 'here', 'and', "I'm", 'actually', 'having', 'fun', 'making', 'this', 'course', 'oI', 'hope', 'you', 'are', 'getting', 'along', 'fine', 'with', 'this', 'presentation', 'I', 'really', 'did', 'try', 'And', 'one', 'last', 'sentence', 'just', 'so', 'you', 'can', 'test', 'you', 'tokenizers', 'better']


## Exercise 2. Get tags from Trump speech using NLTK

You should use the ``trump.txt`` file, read it and find the tags for each word. Use NLTK for it.

In [10]:
import nltk
from nltk.tokenize import word_tokenize

# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('averaged_perceptron_tagger_eng')

file = open("./datasets/trump.txt", "r",encoding="utf-8") 
trump = file.read()
words = word_tokenize(trump)
file.close()

tags = nltk.pos_tag(words)
print(tags[:20])

[('Thank', 'NNP'), ('you', 'PRP'), ('very', 'RB'), ('much', 'RB'), ('.', '.'), ('Mr.', 'NNP'), ('Speaker', 'NNP'), (',', ','), ('Mr.', 'NNP'), ('Vice', 'NNP'), ('President', 'NNP'), (',', ','), ('Members', 'NNP'), ('of', 'IN'), ('Congress', 'NNP'), (',', ','), ('the', 'DT'), ('First', 'NNP'), ('Lady', 'NNP'), ('of', 'IN')]


## Exercise 3. Get the nouns in the last 10 sentences from Trump's speech and find the nouns divided by sentencens. Use SpaCy.

Please use Python list features to get the last 10 sentences and display nouns from it.

In [22]:
import spacy

file = open("./datasets/trump.txt", "r",encoding='utf-8') 
trump = file.read() 
file.close()

nlp = spacy.load("en_core_web_sm")

doc = nlp(trump)
sentences = list(doc.sents)
last_10_sentences = sentences[-10:]
nouns_by_sentence = []

for sentence in last_10_sentences:
    nouns = []
    for token in sentence:
        if token.pos_ in ["NOUN", "PROPN"]:
            nouns.append(token.text)

    nouns_by_sentence.append(nouns)

for i, nouns in enumerate(nouns_by_sentence, 1):
    print(f"Sentence {i} [number of nouns: {len(nouns)}]: {nouns}")

Sentence 1 [number of nouns: 6]: ['vision', 'years', 'freedom', 'tonight', 'chapter', 'greatness']
Sentence 2 [number of nouns: 2]: ['time', 'thinking']
Sentence 3 [number of nouns: 2]: ['time', 'fights']
Sentence 4 [number of nouns: 10]: ['courage', 'dreams', 'hearts', 'bravery', 'hopes', 'souls', 'confidence', 'hopes', 'dreams', 'action']
Sentence 5 [number of nouns: 8]: ['America', 'aspirations', 'fears', 'future', 'failures', 'past', 'vision', 'doubts']
Sentence 6 [number of nouns: 3]: ['citizens', 'renewal', 'spirit']
Sentence 7 [number of nouns: 4]: ['Members', 'Congress', 'things', 'country']
Sentence 8 [number of nouns: 2]: ['tonight', 'moment']
Sentence 9 [number of nouns: 3]: ['yourselves', 'future', 'America']
Sentence 10 [number of nouns: 4]: ['God', 'God', 'United', 'States']


## Exercise 4. Build your own Bag Of Words implementation using tokenizer created before 

You need to implement following methods:

- ``fit_transform`` - gets a list of strings and returns matrix with it's BoW representation
- ``get_features_names`` - returns list of words corresponding to columns in BoW

In [24]:
import numpy as np
import spacy

class BagOfWords:
    """Basic BoW implementation."""
    
    __nlp = spacy.load("en_core_web_sm")
    __bow_list = []
    
    
    def fit_transform(self, corpus: list):
        """Transform list of strings into BoW array.

        Parameters
        ----------
        corpus: List[str]
                Corpus of texts to be transforrmed

        Returns
        -------
        np.array
                Matrix representation of BoW

        """
        tokenized_corpus = []
        vocab = set()
        
        for document in corpus:
            tokens = tokenize_words(document.lower())
            tokenized_corpus.append(tokens)
            vocab.update(tokens)
            
        self.__bow_list = sorted(list(vocab))
        matrix = np.zeros((len(corpus), len(self.__bow_list)), dtype=int)

        for i, tokens in enumerate(tokenized_corpus):
            for word in tokens:
                col_index = self.__bow_list.index(word)
                matrix[i, col_index] += 1
                
        return matrix
      

    def get_feature_names(self) -> list:
        """Return words corresponding to columns of matrix.

        Returns
        -------
        List[str]
                Words being transformed by fit function

        """ 
        return self.__bow_list

corpus = [
     'Bag Of Words is based on counting',
     'words occurences throughout multiple documents.',
     'This is the third document.',
     'As you can see most of the words occur only once.',
     'This gives us a pretty sparse matrix, see below. Really, see below',
]    
    
vectorizer = BagOfWords()

X = vectorizer.fit_transform(corpus)
print(X)

print(f"Feature names: {vectorizer.get_feature_names()}")
print(f"Number of features: {len(vectorizer.get_feature_names())}")

[[0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1]
 [1 0 0 0 2 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 2 1 0 0 1 0 1 0 0]]
Feature names: ['a', 'as', 'bag', 'based', 'below', 'can', 'counting', 'document', 'documents', 'gives', 'is', 'matrix', 'most', 'multiple', 'occur', 'occurences', 'of', 'on', 'once', 'only', 'pretty', 'really', 'see', 'sparse', 'the', 'third', 'this', 'throughout', 'us', 'words', 'you']
Number of features: 31


## Exercise 5. Build a 5-gram model and clean up the results.

There are three tasks to do:
1. Use 5-gram model instead of 3.
2. Change to capital letter each first letter of a sentence.
3. Remove the whitespace between the last word in a sentence and . ! or ?.

Hint: for 2. and 3. implement a function called ``clean_generated()`` that takes the generated text and fix both issues at once. It could be easier to fix the text after it's generated rather then doing some changes in the while loop.

In [ ]:
from nltk.book import *
import random 
wall_street = text7.tokens

import re

tokens = wall_street

def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    clean = list(filter(compiled_pattern.match,tokens))
    return clean
tokens = cleanup()

def build_ngrams():
    ngrams = []
    for i in range(len(tokens)-N+1):
        ngrams.append(tokens[i:i+N])
    return ngrams

def ngram_freqs(ngrams):
    counts = {}

    for ngram in ngrams:
        token_seq  = SEP.join(ngram[:-1])
        last_token = ngram[-1]

        if token_seq not in counts:
            counts[token_seq] = {}

        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0

        counts[token_seq][last_token] += 1

    return counts

def next_word(text, N, counts):

    token_seq = SEP.join(text.split()[-(N-1):])
    choices = counts[token_seq].items()

    total = sum(weight for choice, weight in choices)
    r = random.uniform(0, total)
    upto = 0
    for choice, weight in choices:
        upto += weight
        if upto > r: return choice
    assert False # should not reach here

In [ ]:
def clean_generated():
    # put your code here
    pass

N=3 # fix it for other value of N

SEP=" "

sentence_count=5

ngrams = build_ngrams()
start_seq="We have"

counts = ngram_freqs(ngrams)

if start_seq is None: start_seq = random.choice(list(counts.keys()))
generated = start_seq.lower()

sentences = 0
while sentences < sentence_count:
    generated += SEP + next_word(generated, N, counts)
    sentences += 1 if generated.endswith(('.','!', '?')) else 0

# put your code here:

print(generated)